# Geomarketing: expansão de lojas do RJ para São Paulo

**Autor: Thiago Barral.** Estudo de faturamento e potencial comercial por bairro.

Esta revisão substitui a execução dependente de Google Drive por um fluxo local reproduzível. O notebook histórico permanece no histórico Git. Consulte o README para protocolo de avaliação e limitações.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "DadosDesafioCientista.xlsx").exists():
    raise FileNotFoundError("Abra o notebook a partir da raiz do repositório.")
sys.path.insert(0, str(ROOT / "scripts"))
from train import FEATURES, load_data, run


## 1. Dados e qualidade

São 160 bairros do RJ com alvos observados e 296 de SP sem alvos. Valores de renda inválidos são convertidos em ausentes e imputados pela mediana aprendida somente no treino de cada divisão. Não interpolamos bairros pela ordem das linhas.

In [ ]:
rj, sp = load_data(ROOT / "DadosDesafioCientista.xlsx")
print({"bairros_rj": len(rj), "bairros_sp": len(sp)})
print("Rendas ausentes:", {"RJ": int(rj.rendaMedia.isna().sum()), "SP": int(sp.rendaMedia.isna().sum())})
rj[FEATURES + ["faturamento"]].describe()

## 2. Seleção, avaliação e previsão

Uma divisão estratificada reserva 30% do RJ para teste. Os candidatos e hiperparâmetros são comparados em validação cruzada de quatro partes nos 70% restantes. A regressão usa RMSE e a classificação usa F1 macro. O teste é avaliado somente para os modelos selecionados. Depois, os modelos são reajustados nos 160 bairros do RJ para estimar SP.

Não há rótulos de SP para avaliar a transferência geográfica. As estimativas não são faturamento observado nem garantias comerciais.

In [ ]:
metrics, predictions = run(ROOT / "DadosDesafioCientista.xlsx", ROOT / "outputs")
predictions.head()

## 3. Resultados por classe

A ordem dos rótulos é explícita para evitar atribuir métricas às classes erradas. Os resultados são da nova execução e não devem ser combinados com as exportações históricas.

In [ ]:
import pandas as pd
pd.DataFrame(metrics["classification"]["report"]).T

In [ ]:
pd.DataFrame(metrics["classification"]["confusion_matrix"],
             index=metrics["classification"]["labels"],
             columns=metrics["classification"]["labels"])

## 4. Limitações e próximos passos

- A amostra de teste tem apenas 48 bairros do RJ.
- A seleção por divisão aleatória pode superestimar a generalização espacial.
- Registros com população zero são sinalizados e precisam de revisão antes de decisões.
- A origem, data de referência, período e unidade do faturamento e regras de potencial precisam de documentação.
- Um próximo passo é validar em blocos geográficos e obter resultados observados de SP.
- A existência de previsões não comprova retorno financeiro de uma nova loja.